# NS-RAG pipeline -- MIMIC-III Clinical Database DEMO run (real patients, open access)

This notebook runs the NS-RAG pipeline on the **MIMIC-III Clinical Database Demo**: 100 REAL de-identified ICU patients, openly licensed (Open Data Commons ODbL v1.0), **no CITI training or data use agreement required**.
https://physionet.org/content/mimiciii-demo/

**What this is:** real (if small-scale, n=100) patient diagnoses, medications, and labs -- not simulated. A genuine step up from the fully-synthetic Synthea pilot.

**What this is NOT:** a substitute for the full credentialed MIMIC-III results (not available to us). The demo excludes free-text discharge notes (stripped for the public release), so notes are synthesized from the real structured facts (see `nsrag/data/mimic_demo_instances.py`), and questions/reference assertions are templated rather than clinician-authored -- same documented limitation as the Synthea pilot, but now grounded in real diagnoses and medications rather than a simulator's output.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine, free tier).

## 1. Download the MIMIC-III Demo (open access, no login needed for the files themselves)

In [ ]:
BASE = 'https://physionet.org/files/mimiciii-demo/1.4'
files = ['PATIENTS.csv', 'ADMISSIONS.csv', 'DIAGNOSES_ICD.csv', 'D_ICD_DIAGNOSES.csv',
         'PRESCRIPTIONS.csv', 'LABEVENTS.csv', 'D_LABITEMS.csv', 'NOTEEVENTS.csv']

!mkdir -p mimic-iii-demo
for fn in files:
    !wget -q -c -O mimic-iii-demo/{fn} {BASE}/{fn}

# Sanity check: a real CSV should start with a comma-separated header,
# NOT with '<' (which would mean we downloaded an HTML/login page by mistake).
import os
for fn in files:
    path = f'mimic-iii-demo/{fn}'
    size = os.path.getsize(path)
    with open(path, 'r', errors='ignore') as f:
        first_line = f.readline().strip()
    ok = 'OK' if (size == 0 or not first_line.startswith('<')) else 'FAILED (got HTML, not CSV!)'
    print(f'{fn:25s} {size:>10} bytes  {ok:30s} first line: {first_line[:60]}')

## 2. Get the NS-RAG pipeline code

In [ ]:
!git clone -q https://github.com/ANONYMOUS/REPO.git nsrag-pipeline
%cd nsrag-pipeline
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, 'src')

## 3. Build QA instances from real MIMIC-III Demo patients

In [ ]:
from nsrag.data.mimic_loader import MimicLoader
from nsrag.data.mimic_demo_instances import build_qa_instances_from_mimic
import pandas as pd

loader = MimicLoader('../mimic-iii-demo')
patients = pd.read_csv('../mimic-iii-demo/PATIENTS.csv')['SUBJECT_ID'].astype(str).unique().tolist()
print(f'{len(patients)} real patients available in the demo')

instances = build_qa_instances_from_mimic(loader, patients)
print(f'{len(instances)} QA instances built (patients with at least one structured event)')
print('Example question:', instances[0].question)
print('Example note (synthesized from real structured data):', instances[0].notes[0].text[:200])
print('Example reference assertions:', instances[0].reference_assertions)

## 4. Run the real pipeline (Flan-T5-Large on GPU)
Same generator and baselines as the synthetic pilot -- only the data source changes.

In [ ]:
from nsrag.generation.generator import HFGenerator
from nsrag.baselines.llm_only import LLMOnly
from nsrag.baselines.textual_rag import TextualRAG
from nsrag.baselines.graph_linearization import GraphLinearization
from nsrag.baselines.ns_rag import NSRAGWithoutVerification, NSRAG
from nsrag.pipeline.run_qa import run_qa_experiment

generator = HFGenerator(model_name='google/flan-t5-large', device='cuda', temperature=0.2)

methods = [
    LLMOnly(generator),
    TextualRAG(generator),
    GraphLinearization(generator),
    NSRAGWithoutVerification(generator),
    NSRAG(generator),
]

results = run_qa_experiment(methods, instances)

## 5. GraphRAG and LightRAG (B4/B5) -- optional, needs extra setup
Requires `pip install graphrag lightrag-hku` and, for GraphRAG, an LLM API key for its indexing step. Uncomment and configure once you have these; see `nsrag/baselines/graph_rag_family.py` docstrings for exact requirements.

In [ ]:
# from nsrag.baselines.graph_rag_family import GraphRAGBaseline, LightRAGBaseline
# graphrag = GraphRAGBaseline(generator, index_dir='path/to/prebuilt/index')
# lightrag = LightRAGBaseline(generator, working_dir='path/to/ingested/dir')
# results.update(run_qa_experiment([graphrag, lightrag], instances))

## 6. Ablation and efficiency (same as before, now on real patients)

In [ ]:
from nsrag.pipeline.run_ablation import run_ablation_experiment
from nsrag.pipeline.run_efficiency import run_efficiency_experiment

ablation_results = run_ablation_experiment(generator, instances)
efficiency = run_efficiency_experiment(generator, instances)

## 7. Results as LaTeX rows

In [ ]:
from nsrag.utils.results_to_latex import qa_results_to_latex, ablation_results_to_latex, efficiency_results_to_latex

row_order = ['B1 - LLM', 'B2 - Textual RAG', 'B3 - Graph Linearization',
             'B7 - NS-RAG w/o Verification', 'B8 - NS-RAG']
print('--- MIMIC-III Demo QA results (real patients, n=%d) ---' % len(instances))
print(qa_results_to_latex(results, row_order))
print()
print('--- MIMIC-III Demo ablation ---')
ablation_row_order = [k for k in ablation_results.keys() if k != '_efficiency']
print(ablation_results_to_latex(ablation_results, ablation_row_order))
print()
print('--- Efficiency (real GPU timing) ---')
print(efficiency_results_to_latex(efficiency))

## 8. (Optional) Save results to your Google Drive

In [ ]:
import json
from google.colab import drive
drive.mount('/content/drive')

out = {
    'qa': {k: (v if k == '_efficiency' else {
        'fact_f1': v.fact_f1, 'cr': v.cr, 'ur': v.ur, 'ec': v.ec}) for k, v in results.items()},
    'ablation': {k: (v if k == '_efficiency' else {
        'fact_f1': v.fact_f1, 'cr': v.cr, 'ur': v.ur, 'ec': v.ec}) for k, v in ablation_results.items()},
    'efficiency': efficiency,
}
with open('/content/drive/MyDrive/nsrag_mimic_demo_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
print('Saved to Google Drive: nsrag_mimic_demo_results.json')